<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/03_NeuroFHIR_QC_Imaging_Data_Preparation_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/03_NeuroFHIR_QC_Imaging_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 03
## Public Brain-MRI Data Preparation and Reference-Mask Integrity

**Notebook filename:** `03_NeuroFHIR_QC_Imaging_Data_Preparation.ipynb`  
**Project root:** `/content/drive/MyDrive/neurofhir-qc`  
**Imaging source:** Medical Segmentation Decathlon, `Task01_BrainTumour`  
**Data policy:** public de-identified imaging + synthetic FHIR R4 context only

### What this notebook builds

This notebook prepares the first real imaging layer for the three locked NeuroFHIR-QC demonstration cases:

1. **Stable**
2. **Progression**
3. **Low confidence**

It:

- enforces the successful Notebook 02 evidence gate;
- discovers the uncompressed public MSD brain-tumor objects in the AWS Open Data mirror;
- evaluates a deterministic subset of reference masks and selects three demonstration cases;
- downloads only the selected public 4-modal MRI volumes and their expert reference masks;
- standardizes FLAIR, T1, T1-contrast, and T2 NIfTI files;
- creates multiclass and whole-tumor binary reference masks;
- validates shape, affine, spacing, label values, non-empty tumor content, and voxel-based reference volume;
- links each prepared imaging case to the corresponding synthetic FHIR patient context;
- creates visual previews, reusable imaging-adapter code, checksums, and an execution audit.

### Important interpretation

The selected MSD images are **public de-identified research images**. The Patient, Condition, dates, and prior volume Observations created in Notebook 01 are **synthetic demonstration context**. The mapping does **not** claim that the public image donor is the synthetic FHIR patient, and it does not claim that the public cases are true longitudinal scans of one person.

The public image is used as the model/evaluation input for the synthetic workflow. Notebook 03 does not create an AI result, does not run segmentation inference, does not calculate the final QC classification, and does not write an AI result to FHIR.

### Public source and citation

- Medical Segmentation Decathlon: https://medicaldecathlon.com/
- AWS Open Data registry: https://registry.opendata.aws/msd/
- Dataset paper: Antonelli et al., *Nature Communications* (2022), DOI `10.1038/s41467-022-30695-9`
- License recorded by the AWS Open Data registry: `CC-BY-SA-4.0`

### Before running

Notebook 02 must have successful execution evidence. Save this notebook into:

`/content/drive/MyDrive/neurofhir-qc/notebooks/03_NeuroFHIR_QC_Imaging_Data_Preparation.ipynb`

before rerunning the final audit cell.

In [1]:
# Cell 1 — Mount Drive and enforce the Notebook 02 evidence gate

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab.") from exc

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"

NOTEBOOK_02_AUDIT_PATH = (
    PROJECT_ROOT / "evaluation/results/notebook_02_hapi_server_read_audit.json"
)
DEMO_CASE_MANIFEST_PATH = (
    PROJECT_ROOT / "data/synthetic_fhir/notebook_01/demo_case_manifest.json"
)

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.write("\n")
    temp_path.replace(path)

def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def notebook_entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            value = manifest.get(key)
            if isinstance(value, list):
                return value
    raise ValueError("Unrecognized notebook_manifest.json structure.")

def normalize_number(value: Any) -> str:
    match = re.search(r"\d+", str(value))
    return match.group(0).zfill(2) if match else str(value)

def find_entry(manifest: Any, number: str) -> dict[str, Any]:
    target = normalize_number(number)
    for entry in notebook_entries(manifest):
        candidates = [
            entry.get("number"),
            entry.get("notebook_number"),
            entry.get("id"),
            entry.get("filename"),
        ]
        if any(
            normalize_number(value) == target
            for value in candidates
            if value is not None
        ):
            return entry
    raise KeyError(f"Notebook {target} is missing from the manifest.")

required_inputs = [
    CONFIG_PATH,
    NOTEBOOK_MANIFEST_PATH,
    NOTEBOOK_02_AUDIT_PATH,
    DEMO_CASE_MANIFEST_PATH,
]
missing = [
    str(path)
    for path in required_inputs
    if not path.exists() or path.stat().st_size == 0
]
if missing:
    raise FileNotFoundError(
        "Notebook 02 / case evidence is incomplete:\n"
        + "\n".join(f" - {path}" for path in missing)
    )

project_config = load_json(CONFIG_PATH)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
notebook_02_audit = load_json(NOTEBOOK_02_AUDIT_PATH)
demo_case_manifest = load_json(DEMO_CASE_MANIFEST_PATH)

if not notebook_02_audit.get("metrics"):
    raise RuntimeError("Notebook 02 audit does not contain measured execution evidence.")

required_notebook_02_metrics = [
    "transaction_success_rate",
    "direct_read_success_rate",
    "critical_signature_match_rate",
    "patient_context_success_rate",
    "server_reference_integrity_rate",
]
failed_notebook_02_metrics = [
    metric
    for metric in required_notebook_02_metrics
    if float(notebook_02_audit["metrics"].get(metric, 0.0)) != 1.0
]
if failed_notebook_02_metrics:
    raise RuntimeError(
        "Notebook 02 completion metrics did not pass: "
        + ", ".join(failed_notebook_02_metrics)
    )

notebook_02_entry = find_entry(notebook_manifest, "02")
notebook_03_entry = find_entry(notebook_manifest, "03")

notebook_02_status = str(
    notebook_02_entry.get("status", notebook_02_audit.get("status", ""))
).lower()
accepted_statuses = {
    "completed",
    "complete",
    "passed",
    "executed_pending_notebook_save",
}
if notebook_02_status not in accepted_statuses:
    raise RuntimeError(
        f"Notebook 02 is not ready. Current status: {notebook_02_status!r}"
    )

case_entries = demo_case_manifest.get("cases", [])
case_ids = [case.get("case_id") for case in case_entries]
if set(case_ids) != {"stable", "progression", "low-confidence"}:
    raise AssertionError("The three locked demonstration cases are not present.")

NOTEBOOK_FILENAME = notebook_03_entry.get(
    "filename",
    "03_NeuroFHIR_QC_Imaging_Data_Preparation.ipynb",
)
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

IMAGE_ROOT = PROJECT_ROOT / "data/sample_images/notebook_03"
MASK_ROOT = PROJECT_ROOT / "data/sample_masks/notebook_03"
EVALUATION_ROOT = (
    PROJECT_ROOT / "evaluation/results/notebook_03_imaging_preparation"
)
PREVIEW_ROOT = EVALUATION_ROOT / "previews"
LOCAL_STAGING_ROOT = Path("/content/neurofhir_qc_notebook03_staging")

DATASET_METADATA_PATH = IMAGE_ROOT / "msd_task01_dataset.json"
SOURCE_INVENTORY_PATH = EVALUATION_ROOT / "source_inventory.json"
SELECTION_REPORT_PATH = EVALUATION_ROOT / "case_selection_report.json"
SELECTION_REPORT_CSV_PATH = EVALUATION_ROOT / "case_selection_report.csv"
IMAGING_MANIFEST_PATH = IMAGE_ROOT / "imaging_case_manifest.json"
IMAGING_INDEX_CSV_PATH = IMAGE_ROOT / "imaging_case_index.csv"
INTEGRITY_REPORT_PATH = EVALUATION_ROOT / "imaging_integrity_report.json"
INTEGRITY_REPORT_CSV_PATH = EVALUATION_ROOT / "imaging_integrity_report.csv"
AUDIT_JSON_PATH = (
    PROJECT_ROOT / "evaluation/results/notebook_03_imaging_preparation_audit.json"
)
AUDIT_MD_PATH = (
    PROJECT_ROOT / "docs/NOTEBOOK_03_IMAGING_DATA_PREPARATION.md"
)

for directory in (
    IMAGE_ROOT,
    MASK_ROOT,
    EVALUATION_ROOT,
    PREVIEW_ROOT,
    LOCAL_STAGING_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 92)
print("✅ Notebook 02 evidence gate passed")
print(f"✅ Notebook 02 status: {notebook_02_status}")
print(f"✅ Locked cases: {sorted(case_ids)}")
print(f"📓 Notebook 03 target path: {NOTEBOOK_SAVE_PATH}")
print("=" * 92)

Mounted at /content/drive
✅ Notebook 02 evidence gate passed
✅ Notebook 02 status: executed_pending_notebook_save
✅ Locked cases: ['low-confidence', 'progression', 'stable']
📓 Notebook 03 target path: /content/drive/MyDrive/neurofhir-qc/notebooks/03_NeuroFHIR_QC_Imaging_Data_Preparation.ipynb


In [2]:
# Cell 2 — Install and record the imaging-data dependencies

required_packages = [
    "boto3>=1.34,<2",
    "nibabel>=5.2,<6",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *required_packages,
    ]
)

import importlib.metadata as importlib_metadata
import itertools

import boto3
import matplotlib
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from botocore import UNSIGNED
from botocore.client import Config as BotoConfig

dependency_versions = {
    "python": sys.version.split()[0],
    "boto3": importlib_metadata.version("boto3"),
    "botocore": importlib_metadata.version("botocore"),
    "nibabel": importlib_metadata.version("nibabel"),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
}

for package, version in dependency_versions.items():
    print(f"{package}: {version}")

print("=" * 92)
print("✅ Imaging dependencies installed and imported")
print("=" * 92)

python: 3.12.13
boto3: 1.43.63
botocore: 1.43.63
nibabel: 5.4.2
numpy: 2.0.2
pandas: 2.2.2
matplotlib: 3.10.0
✅ Imaging dependencies installed and imported


In [3]:
# Cell 3 — Discover MSD Task01 members inside the public S3 TAR archive

AWS_BUCKET = "msd-for-monai"
AWS_REGION = "us-west-2"
ARCHIVE_KEY = "Task01_BrainTumour.tar"
DATASET_PREFIX = "Task01_BrainTumour/"  # virtual path inside the TAR archive
DATASET_NAME = "Medical Segmentation Decathlon — Task01 BrainTumour"
DATASET_LICENSE = "CC-BY-SA-4.0"
DATASET_DOI = "10.1038/s41467-022-30695-9"
AWS_REGISTRY_URL = "https://registry.opendata.aws/msd/"
MSD_URL = "https://medicaldecathlon.com/"
TAR_INDEX_PATH = EVALUATION_ROOT / "task01_brain_tumour_tar_index.json"

# The public bucket exposes Task01 primarily as one uncompressed TAR object.
# Therefore, do not try to list Task01_BrainTumour/ as an S3 prefix. Instead,
# index the TAR headers with small unsigned byte-range requests, then download
# only dataset.json, the candidate labels, and the three selected MRI files.
archive_candidates = [
    ("msd-for-monai", "us-west-2"),
    ("msd-for-monai-eu", "eu-west-2"),
]

s3 = None
archive_head = None
connection_errors: list[str] = []
for candidate_bucket, candidate_region in archive_candidates:
    candidate_client = boto3.client(
        "s3",
        region_name=candidate_region,
        config=BotoConfig(
            signature_version=UNSIGNED,
            retries={"max_attempts": 5, "mode": "standard"},
        ),
    )
    try:
        candidate_head = candidate_client.head_object(
            Bucket=candidate_bucket,
            Key=ARCHIVE_KEY,
        )
    except Exception as exc:
        connection_errors.append(
            f"s3://{candidate_bucket}/{ARCHIVE_KEY}: "
            f"{type(exc).__name__}: {exc}"
        )
        continue

    s3 = candidate_client
    AWS_BUCKET = candidate_bucket
    AWS_REGION = candidate_region
    archive_head = candidate_head
    break

if s3 is None or archive_head is None:
    raise RuntimeError(
        "Could not access the public MSD Task01 TAR archive. Attempts:\n"
        + "\n".join(f" - {message}" for message in connection_errors)
    )

archive_size_bytes = int(archive_head["ContentLength"])
archive_etag = str(archive_head.get("ETag", "")).strip('"')
if archive_size_bytes <= 1024:
    raise RuntimeError(
        f"Unexpected archive size for s3://{AWS_BUCKET}/{ARCHIVE_KEY}: "
        f"{archive_size_bytes} bytes"
    )


def read_s3_range(start: int, end: int) -> bytes:
    if start < 0 or end < start or end >= archive_size_bytes:
        raise ValueError(f"Invalid byte range: {start}-{end}")
    response = s3.get_object(
        Bucket=AWS_BUCKET,
        Key=ARCHIVE_KEY,
        Range=f"bytes={start}-{end}",
    )
    payload = response["Body"].read()
    expected = end - start + 1
    if len(payload) != expected:
        raise IOError(
            f"Incomplete S3 range read {start}-{end}: "
            f"received {len(payload)} of {expected} bytes"
        )
    return payload


def decode_tar_text(raw: bytes) -> str:
    return raw.split(b"\0", 1)[0].decode("utf-8", errors="replace").strip()


def parse_tar_size(raw: bytes) -> int:
    # POSIX TAR normally stores size as an ASCII octal value. GNU TAR may use
    # base-256 for very large values, so handle both representations.
    if raw and (raw[0] & 0x80):
        value = int.from_bytes(raw, byteorder="big", signed=True)
        return value & ((1 << (len(raw) * 8 - 1)) - 1)
    text = raw.rstrip(b"\0 ").lstrip(b" ")
    return int(text or b"0", 8)


def normalize_tar_member_name(name: str) -> str:
    normalized = name.replace("\\", "/")
    while normalized.startswith("./"):
        normalized = normalized[2:]
    return normalized.lstrip("/")


def build_tar_index() -> list[dict[str, Any]]:
    members: list[dict[str, Any]] = []
    offset = 0
    zero_block_count = 0
    pending_long_name: str | None = None

    while offset + 512 <= archive_size_bytes:
        header = read_s3_range(offset, offset + 511)
        if header == b"\0" * 512:
            zero_block_count += 1
            offset += 512
            if zero_block_count >= 2:
                break
            continue

        zero_block_count = 0
        name = decode_tar_text(header[0:100])
        prefix = decode_tar_text(header[345:500])
        if prefix:
            name = f"{prefix}/{name}" if name else prefix
        name = normalize_tar_member_name(name)

        size = parse_tar_size(header[124:136])
        typeflag = header[156:157] or b"0"
        data_offset = offset + 512
        padded_size = ((size + 511) // 512) * 512
        next_header_offset = data_offset + padded_size

        if next_header_offset > archive_size_bytes + 512:
            raise RuntimeError(
                f"Invalid TAR member boundary at offset {offset}: "
                f"name={name!r}, size={size}"
            )

        if typeflag == b"L":
            # GNU long-name record applying to the next member.
            if size > 0:
                long_name_payload = read_s3_range(
                    data_offset,
                    data_offset + size - 1,
                )
                pending_long_name = normalize_tar_member_name(
                    long_name_payload.rstrip(b"\0\n").decode(
                        "utf-8",
                        errors="replace",
                    )
                )
        elif typeflag in (b"0", b"\0"):
            effective_name = pending_long_name or name
            pending_long_name = None
            members.append(
                {
                    "name": effective_name,
                    "size_bytes": int(size),
                    "header_offset": int(offset),
                    "data_offset": int(data_offset),
                    "data_end_offset": int(data_offset + size - 1)
                    if size > 0
                    else int(data_offset - 1),
                }
            )
        else:
            pending_long_name = None

        offset = next_header_offset

    if not members:
        raise RuntimeError(
            f"No regular files were indexed inside "
            f"s3://{AWS_BUCKET}/{ARCHIVE_KEY}"
        )
    return members


cached_tar_index = None
if TAR_INDEX_PATH.exists() and TAR_INDEX_PATH.stat().st_size > 0:
    try:
        candidate_index = load_json(TAR_INDEX_PATH)
        if (
            candidate_index.get("archive_bucket") == AWS_BUCKET
            and candidate_index.get("archive_key") == ARCHIVE_KEY
            and int(candidate_index.get("archive_size_bytes", -1))
            == archive_size_bytes
            and candidate_index.get("archive_etag") == archive_etag
            and isinstance(candidate_index.get("members"), list)
            and candidate_index["members"]
        ):
            cached_tar_index = candidate_index
    except Exception:
        cached_tar_index = None

if cached_tar_index is None:
    print(
        "Indexing TAR headers with small unsigned S3 range requests. "
        "This does not download the full archive."
    )
    tar_members = build_tar_index()
    write_json(
        TAR_INDEX_PATH,
        {
            "generated_utc": utc_now(),
            "archive_bucket": AWS_BUCKET,
            "archive_region": AWS_REGION,
            "archive_key": ARCHIVE_KEY,
            "archive_size_bytes": archive_size_bytes,
            "archive_etag": archive_etag,
            "member_count": len(tar_members),
            "members": tar_members,
        },
    )
else:
    tar_members = cached_tar_index["members"]
    print(f"Using cached TAR index: {TAR_INDEX_PATH}")


tar_member_by_name = {
    normalize_tar_member_name(member["name"]): member
    for member in tar_members
}

dataset_json_keys = sorted(
    name
    for name in tar_member_by_name
    if name.endswith("Task01_BrainTumour/dataset.json")
)
image_keys = sorted(
    name
    for name in tar_member_by_name
    if "/imagesTr/" in name and name.endswith(".nii.gz")
)
label_keys = sorted(
    name
    for name in tar_member_by_name
    if "/labelsTr/" in name and name.endswith(".nii.gz")
)

if not dataset_json_keys:
    raise FileNotFoundError("MSD Task01 dataset.json was not found in the TAR archive.")
if len(image_keys) < 3 or len(label_keys) < 3:
    raise RuntimeError(
        f"Insufficient training members: {len(image_keys)} images, "
        f"{len(label_keys)} labels."
    )


def nifti_stem(key: str) -> str:
    name = Path(key).name
    return name[:-7] if name.endswith(".nii.gz") else Path(name).stem


images_by_stem = {nifti_stem(key): key for key in image_keys}
labels_by_stem = {nifti_stem(key): key for key in label_keys}
paired_stems = sorted(set(images_by_stem) & set(labels_by_stem))
if len(paired_stems) < 3:
    raise RuntimeError("Fewer than three paired MSD image/mask cases were found.")


def download_tar_member(member_name: str, destination: Path) -> dict[str, Any]:
    normalized_name = normalize_tar_member_name(member_name)
    member = tar_member_by_name.get(normalized_name)
    if member is None:
        raise FileNotFoundError(
            f"TAR member not found: {normalized_name}"
        )

    expected_size = int(member["size_bytes"])
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size == expected_size:
        return {
            "key": normalized_name,
            "archive_key": ARCHIVE_KEY,
            "size_bytes": expected_size,
            "archive_etag": archive_etag,
            "range_start": int(member["data_offset"]),
            "range_end": int(member["data_end_offset"]),
            "sha256": sha256_file(destination),
            "local_path": str(destination),
            "cached": True,
        }

    temporary = destination.with_suffix(destination.suffix + ".part")
    if temporary.exists():
        temporary.unlink()

    if expected_size == 0:
        temporary.write_bytes(b"")
    else:
        response = s3.get_object(
            Bucket=AWS_BUCKET,
            Key=ARCHIVE_KEY,
            Range=(
                f"bytes={int(member['data_offset'])}-"
                f"{int(member['data_end_offset'])}"
            ),
        )
        with temporary.open("wb") as handle:
            while True:
                chunk = response["Body"].read(1024 * 1024)
                if not chunk:
                    break
                handle.write(chunk)

    if temporary.stat().st_size != expected_size:
        raise IOError(
            f"Extracted size mismatch for {normalized_name}: "
            f"{temporary.stat().st_size} != {expected_size}"
        )
    temporary.replace(destination)

    return {
        "key": normalized_name,
        "archive_key": ARCHIVE_KEY,
        "size_bytes": expected_size,
        "archive_etag": archive_etag,
        "range_start": int(member["data_offset"]),
        "range_end": int(member["data_end_offset"]),
        "sha256": sha256_file(destination),
        "local_path": str(destination),
        "cached": False,
    }


dataset_json_key = dataset_json_keys[0]
download_tar_member(dataset_json_key, DATASET_METADATA_PATH)
dataset_metadata = load_json(DATASET_METADATA_PATH)

modality_map_raw = dataset_metadata.get("modality", {})
modality_map = {
    int(index): str(name)
    for index, name in modality_map_raw.items()
}
label_map_raw = dataset_metadata.get("labels", {})
label_map = {
    int(index): str(name)
    for index, name in label_map_raw.items()
}
if len(modality_map) != 4:
    raise AssertionError(
        f"Expected four MRI modalities; found {modality_map!r}"
    )

candidate_count = min(36, len(paired_stems))
candidate_indices = sorted(
    {
        int(round(index))
        for index in np.linspace(
            0,
            len(paired_stems) - 1,
            num=candidate_count,
        )
    }
)
candidate_stems = [paired_stems[index] for index in candidate_indices]

source_inventory = {
    "generated_utc": utc_now(),
    "dataset_name": DATASET_NAME,
    "bucket": AWS_BUCKET,
    "region": AWS_REGION,
    "prefix": DATASET_PREFIX,
    "archive_key": ARCHIVE_KEY,
    "archive_size_bytes": archive_size_bytes,
    "archive_etag": archive_etag,
    "source_access_mode": (
        "Unsigned S3 byte-range extraction from the uncompressed TAR archive"
    ),
    "tar_index_path": TAR_INDEX_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "dataset_json_key": dataset_json_key,
    "license": DATASET_LICENSE,
    "citation_doi": DATASET_DOI,
    "aws_registry_url": AWS_REGISTRY_URL,
    "medical_decathlon_url": MSD_URL,
    "training_image_count": len(image_keys),
    "training_label_count": len(label_keys),
    "paired_case_count": len(paired_stems),
    "candidate_case_count": len(candidate_stems),
    "modality_map": modality_map,
    "label_map": label_map,
    "candidate_cases": [
        {
            "source_case_id": stem,
            "image_key": images_by_stem[stem],
            "image_size_bytes": int(
                tar_member_by_name[images_by_stem[stem]]["size_bytes"]
            ),
            "label_key": labels_by_stem[stem],
            "label_size_bytes": int(
                tar_member_by_name[labels_by_stem[stem]]["size_bytes"]
            ),
        }
        for stem in candidate_stems
    ],
}
write_json(SOURCE_INVENTORY_PATH, source_inventory)

print("=" * 92)
print("✅ Public MSD Task01 TAR archive discovered and indexed")
print(f"🪣 Source: s3://{AWS_BUCKET}/{ARCHIVE_KEY}")
print(f"🧠 Paired training cases: {len(paired_stems)}")
print(f"🔎 Candidate labels selected for volume screening: {len(candidate_stems)}")
print(f"🧲 Modalities: {modality_map}")
print(f"🏷️ Labels: {label_map}")
print(f"📄 Source inventory: {SOURCE_INVENTORY_PATH}")
print("=" * 92)


Indexing TAR headers with small unsigned S3 range requests. This does not download the full archive.
✅ Public MSD Task01 TAR archive discovered and indexed
🪣 Source: s3://msd-for-monai/Task01_BrainTumour.tar
🧠 Paired training cases: 485
🔎 Candidate labels selected for volume screening: 36
🧲 Modalities: {0: 'FLAIR', 1: 'T1w', 2: 't1gd', 3: 'T2w'}
🏷️ Labels: {0: 'background', 1: 'edema', 2: 'non-enhancing tumor', 3: 'enhancing tumour'}
📄 Source inventory: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_03_imaging_preparation/source_inventory.json


In [5]:
# Repair cell — exclude macOS AppleDouble metadata from MSD case selection

def is_archive_metadata_key(key: str) -> bool:
    normalized = normalize_tar_member_name(key)
    parts = [part for part in normalized.split("/") if part]
    return any(
        part == "__MACOSX" or part.startswith("._")
        for part in parts
    )


eligible_paired_stems = sorted(
    stem
    for stem in paired_stems
    if (
        not stem.startswith("._")
        and stem in images_by_stem
        and stem in labels_by_stem
        and not is_archive_metadata_key(images_by_stem[stem])
        and not is_archive_metadata_key(labels_by_stem[stem])
    )
)

if len(eligible_paired_stems) < 3:
    raise RuntimeError(
        f"Only {len(eligible_paired_stems)} valid MRI/mask pairs remain."
    )

# Recreate the 36-case screening sample using only genuine NIfTI pairs.
candidate_count = min(36, len(eligible_paired_stems))

candidate_indices = sorted(
    {
        int(round(index))
        for index in np.linspace(
            0,
            len(eligible_paired_stems) - 1,
            num=candidate_count,
        )
    }
)

candidate_stems = [
    eligible_paired_stems[index]
    for index in candidate_indices
]

# Remove the invalid metadata file downloaded during the failed run.
candidate_label_root = (
    LOCAL_STAGING_ROOT / "candidate_labels"
)

if candidate_label_root.exists():
    for stale_file in candidate_label_root.glob("._*.nii.gz"):
        stale_file.unlink(missing_ok=True)
        print(f"Removed invalid metadata file: {stale_file.name}")

print("✅ Archive metadata excluded")
print(f"✅ Valid paired MRI/mask cases: {len(eligible_paired_stems)}")
print(f"✅ Cases selected for screening: {len(candidate_stems)}")
print(f"✅ First selected case: {candidate_stems[0]}")

Removed invalid metadata file: ._BRATS_166.nii.gz
✅ Archive metadata excluded
✅ Valid paired MRI/mask cases: 484
✅ Cases selected for screening: 36
✅ First selected case: BRATS_001


In [6]:
# Cell 4 — Screen candidate masks and select three reproducible demo cases

CANDIDATE_LABEL_ROOT = LOCAL_STAGING_ROOT / "candidate_labels"
SELECTED_IMAGE_ROOT = LOCAL_STAGING_ROOT / "selected_images"
CANDIDATE_LABEL_ROOT.mkdir(parents=True, exist_ok=True)
SELECTED_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)


def download_s3_object(key: str, destination: Path) -> dict[str, Any]:
    # Compatibility wrapper retained for the remainder of this notebook.
    # `key` is a member path inside Task01_BrainTumour.tar, not a standalone
    # S3 object. Only that member's byte range is downloaded.
    return download_tar_member(key, destination)


def reference_mask_summary(path: Path) -> dict[str, Any]:
    image = nib.load(str(path))
    data = np.asanyarray(image.dataobj)
    if data.ndim != 3:
        raise ValueError(f"Reference mask must be 3D: {path}, shape={data.shape}")
    if not np.isfinite(data).all():
        raise ValueError(f"Reference mask contains non-finite values: {path}")
    rounded = np.rint(data).astype(np.int16)
    unique_values = sorted(int(value) for value in np.unique(rounded))
    spacing_mm = tuple(float(value) for value in image.header.get_zooms()[:3])
    voxel_volume_mm3 = float(np.prod(spacing_mm))
    whole_tumor_voxels = int(np.count_nonzero(rounded > 0))
    whole_tumor_volume_ml = whole_tumor_voxels * voxel_volume_mm3 / 1000.0
    return {
        "shape": [int(value) for value in rounded.shape],
        "spacing_mm": [round(value, 6) for value in spacing_mm],
        "unique_labels": unique_values,
        "whole_tumor_voxels": whole_tumor_voxels,
        "whole_tumor_reference_volume_ml": round(
            whole_tumor_volume_ml,
            6,
        ),
    }


candidate_summaries: list[dict[str, Any]] = []
for position, stem in enumerate(candidate_stems, start=1):
    label_key = labels_by_stem[stem]
    local_label = CANDIDATE_LABEL_ROOT / f"{stem}.nii.gz"
    transfer = download_s3_object(label_key, local_label)
    summary = reference_mask_summary(local_label)
    if summary["whole_tumor_voxels"] <= 0:
        raise AssertionError(f"Candidate {stem} has an empty tumor mask.")
    candidate_summaries.append(
        {
            "source_case_id": stem,
            "image_key": images_by_stem[stem],
            "label_key": label_key,
            "label_transfer": transfer,
            **summary,
        }
    )
    print(
        f"[{position:02d}/{len(candidate_stems):02d}] "
        f"{stem}: {summary['whole_tumor_reference_volume_ml']:.2f} mL"
    )

case_order = ["stable", "progression", "low-confidence"]
demo_by_id = {
    case["case_id"]: case
    for case in demo_case_manifest["cases"]
}
target_volume_by_case = {
    case_id: float(
        demo_by_id[case_id]["planned_followup_reference_volume_ml"]
    )
    for case_id in case_order
}

# Evaluate all distinct three-case assignments and choose the minimum
# total absolute difference from the synthetic scenario-design volumes.
best_assignment: tuple[float, tuple[dict[str, Any], ...]] | None = None
for selected in itertools.permutations(candidate_summaries, len(case_order)):
    total_absolute_difference = sum(
        abs(
            selected[index]["whole_tumor_reference_volume_ml"]
            - target_volume_by_case[case_id]
        )
        for index, case_id in enumerate(case_order)
    )
    if (
        best_assignment is None
        or total_absolute_difference < best_assignment[0]
    ):
        best_assignment = (total_absolute_difference, selected)

if best_assignment is None:
    raise RuntimeError("No valid case assignment could be selected.")

selected_summaries = list(best_assignment[1])
selection_rows: list[dict[str, Any]] = []

for case_id, source_summary in zip(case_order, selected_summaries):
    source_case_id = source_summary["source_case_id"]
    image_key = source_summary["image_key"]
    local_image = SELECTED_IMAGE_ROOT / f"{source_case_id}.nii.gz"
    image_transfer = download_s3_object(image_key, local_image)

    selection_rows.append(
        {
            "case_id": case_id,
            "source_case_id": source_case_id,
            "target_scenario_volume_ml": target_volume_by_case[case_id],
            "public_reference_volume_ml": (
                source_summary["whole_tumor_reference_volume_ml"]
            ),
            "absolute_difference_ml": round(
                abs(
                    source_summary["whole_tumor_reference_volume_ml"]
                    - target_volume_by_case[case_id]
                ),
                6,
            ),
            "image_key": image_key,
            "label_key": source_summary["label_key"],
            "image_transfer": image_transfer,
            "label_transfer": source_summary["label_transfer"],
            "label_summary": {
                key: source_summary[key]
                for key in (
                    "shape",
                    "spacing_mm",
                    "unique_labels",
                    "whole_tumor_voxels",
                    "whole_tumor_reference_volume_ml",
                )
            },
        }
    )

selection_report = {
    "generated_utc": utc_now(),
    "selection_method": (
        "Deterministic screening of evenly spaced public training masks, "
        "followed by minimum total absolute volume difference across three "
        "distinct assignments. This is demo-case selection, not a model "
        "performance sample."
    ),
    "candidate_count": len(candidate_summaries),
    "target_volume_interpretation": (
        "Synthetic scenario-design targets are used only to select visually "
        "compatible public examples. They are not asserted to be clinical "
        "measurements from the public-image donors."
    ),
    "total_absolute_difference_ml": round(best_assignment[0], 6),
    "selected_cases": selection_rows,
}
write_json(SELECTION_REPORT_PATH, selection_report)

with SELECTION_REPORT_CSV_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    fieldnames = [
        "case_id",
        "source_case_id",
        "target_scenario_volume_ml",
        "public_reference_volume_ml",
        "absolute_difference_ml",
        "image_key",
        "label_key",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(
        {
            key: row[key]
            for key in fieldnames
        }
        for row in selection_rows
    )

print("=" * 92)
print("✅ Three public MRI/reference-mask pairs selected")
for row in selection_rows:
    print(
        f" - {row['case_id']}: {row['source_case_id']} | "
        f"public reference={row['public_reference_volume_ml']:.2f} mL | "
        f"scenario target={row['target_scenario_volume_ml']:.2f} mL"
    )
print(f"📄 Selection report: {SELECTION_REPORT_PATH}")
print("=" * 92)


[01/36] BRATS_001: 111.72 mL
[02/36] BRATS_015: 198.69 mL
[03/36] BRATS_029: 18.89 mL
[04/36] BRATS_042: 32.57 mL
[05/36] BRATS_056: 148.06 mL
[06/36] BRATS_070: 50.31 mL
[07/36] BRATS_084: 28.39 mL
[08/36] BRATS_098: 89.51 mL
[09/36] BRATS_111: 96.47 mL
[10/36] BRATS_125: 127.23 mL
[11/36] BRATS_139: 85.66 mL
[12/36] BRATS_153: 60.91 mL
[13/36] BRATS_167: 22.58 mL
[14/36] BRATS_180: 72.69 mL
[15/36] BRATS_194: 91.13 mL
[16/36] BRATS_208: 66.75 mL
[17/36] BRATS_222: 78.27 mL
[18/36] BRATS_236: 170.55 mL
[19/36] BRATS_249: 318.35 mL
[20/36] BRATS_263: 165.33 mL
[21/36] BRATS_277: 210.33 mL
[22/36] BRATS_291: 100.89 mL
[23/36] BRATS_305: 55.13 mL
[24/36] BRATS_318: 95.63 mL
[25/36] BRATS_332: 100.88 mL
[26/36] BRATS_346: 45.95 mL
[27/36] BRATS_360: 59.25 mL
[28/36] BRATS_374: 43.46 mL
[29/36] BRATS_387: 150.81 mL
[30/36] BRATS_401: 44.80 mL
[31/36] BRATS_415: 82.92 mL
[32/36] BRATS_429: 58.82 mL
[33/36] BRATS_443: 21.98 mL
[34/36] BRATS_456: 223.00 mL
[35/36] BRATS_470: 127.81 mL
[36/36]

In [7]:
# Cell 5 — Standardize MRI modalities and persist validated reference masks

def safe_component(value: str) -> str:
    normalized = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return normalized or "modality"

def save_nifti(
    data: np.ndarray,
    affine: np.ndarray,
    header: nib.nifti1.Nifti1Header,
    destination: Path,
    dtype: np.dtype,
) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    clean_header = header.copy()
    clean_header.set_data_dtype(dtype)
    output = nib.Nifti1Image(data.astype(dtype, copy=False), affine, clean_header)
    nib.save(output, str(destination))
    if not destination.exists() or destination.stat().st_size == 0:
        raise IOError(f"Failed to write NIfTI file: {destination}")

def infer_channel_axis(
    image_shape: tuple[int, ...],
    modality_count: int,
) -> int:
    if len(image_shape) != 4:
        raise ValueError(
            f"Expected a four-dimensional multimodal MRI; found {image_shape}"
        )
    if image_shape[-1] == modality_count:
        return 3
    if image_shape[0] == modality_count:
        return 0
    raise ValueError(
        f"Could not identify the modality axis in image shape {image_shape} "
        f"for {modality_count} modalities."
    )

allowed_label_values = set(label_map)
prepared_rows: list[dict[str, Any]] = []

for row in selection_rows:
    case_id = row["case_id"]
    source_case_id = row["source_case_id"]
    local_image = Path(row["image_transfer"]["local_path"])
    local_label = Path(row["label_transfer"]["local_path"])

    case_image_root = IMAGE_ROOT / case_id
    case_mask_root = MASK_ROOT / case_id
    if case_image_root.exists():
        shutil.rmtree(case_image_root)
    if case_mask_root.exists():
        shutil.rmtree(case_mask_root)
    modality_root = case_image_root / "modalities"
    modality_root.mkdir(parents=True, exist_ok=True)
    case_mask_root.mkdir(parents=True, exist_ok=True)

    image_nii = nib.load(str(local_image))
    label_nii = nib.load(str(local_label))
    image_data = np.asanyarray(image_nii.dataobj)
    raw_label_data = np.asanyarray(label_nii.dataobj)

    if not np.isfinite(image_data).all():
        raise ValueError(f"{source_case_id} MRI contains non-finite values.")
    if not np.isfinite(raw_label_data).all():
        raise ValueError(f"{source_case_id} mask contains non-finite values.")

    label_data = np.rint(raw_label_data).astype(np.uint8)

    channel_axis = infer_channel_axis(image_data.shape, len(modality_map))
    if channel_axis == 0:
        channel_last = np.moveaxis(image_data, 0, -1)
    else:
        channel_last = image_data

    spatial_shape = tuple(int(value) for value in channel_last.shape[:3])
    if spatial_shape != tuple(int(value) for value in label_data.shape):
        raise AssertionError(
            f"Image/mask shape mismatch for {source_case_id}: "
            f"{spatial_shape} vs {label_data.shape}"
        )

    if not np.allclose(
        image_nii.affine,
        label_nii.affine,
        atol=1e-4,
        rtol=0.0,
    ):
        raise AssertionError(
            f"Image/mask affine mismatch for {source_case_id}."
        )
    if not np.isfinite(image_nii.affine).all():
        raise ValueError(f"Non-finite affine for {source_case_id}.")
    if abs(float(np.linalg.det(image_nii.affine[:3, :3]))) <= 1e-8:
        raise ValueError(f"Singular image affine for {source_case_id}.")

    unique_labels = sorted(int(value) for value in np.unique(label_data))
    unexpected_labels = sorted(set(unique_labels) - allowed_label_values)
    if unexpected_labels:
        raise AssertionError(
            f"Unexpected labels for {source_case_id}: {unexpected_labels}"
        )

    whole_tumor_mask = (label_data > 0).astype(np.uint8)
    whole_tumor_voxels = int(whole_tumor_mask.sum())
    if whole_tumor_voxels <= 0:
        raise AssertionError(f"Empty whole-tumor mask for {source_case_id}.")

    spacing_mm = tuple(
        float(value)
        for value in image_nii.header.get_zooms()[:3]
    )
    voxel_volume_mm3 = float(np.prod(spacing_mm))
    reference_volume_ml = whole_tumor_voxels * voxel_volume_mm3 / 1000.0

    modality_files: dict[str, str] = {}
    for channel_index, modality_name in sorted(modality_map.items()):
        modality_slug = safe_component(modality_name)
        modality_path = modality_root / f"{modality_slug}.nii.gz"
        save_nifti(
            channel_last[..., channel_index],
            image_nii.affine,
            image_nii.header,
            modality_path,
            np.float32,
        )
        modality_files[modality_name] = (
            modality_path.relative_to(PROJECT_ROOT).as_posix()
        )

    multiclass_mask_path = case_mask_root / "reference_multiclass.nii.gz"
    whole_tumor_mask_path = case_mask_root / "whole_tumor_binary.nii.gz"
    save_nifti(
        label_data,
        label_nii.affine,
        label_nii.header,
        multiclass_mask_path,
        np.uint8,
    )
    save_nifti(
        whole_tumor_mask,
        label_nii.affine,
        label_nii.header,
        whole_tumor_mask_path,
        np.uint8,
    )

    case_metadata = {
        "case_id": case_id,
        "source_case_id": source_case_id,
        "source_dataset": DATASET_NAME,
        "source_image_key": row["image_key"],
        "source_label_key": row["label_key"],
        "source_image_sha256": row["image_transfer"]["sha256"],
        "source_label_sha256": row["label_transfer"]["sha256"],
        "spatial_shape": list(spatial_shape),
        "source_image_shape": [int(value) for value in image_data.shape],
        "source_channel_axis": channel_axis,
        "modality_order": [
            modality_map[index]
            for index in sorted(modality_map)
        ],
        "spacing_mm": [round(value, 6) for value in spacing_mm],
        "voxel_volume_mm3": round(voxel_volume_mm3, 9),
        "affine": [
            [round(float(value), 8) for value in affine_row]
            for affine_row in image_nii.affine.tolist()
        ],
        "unique_reference_labels": unique_labels,
        "label_definitions": label_map,
        "whole_tumor_voxels": whole_tumor_voxels,
        "whole_tumor_reference_volume_ml": round(reference_volume_ml, 6),
        "modality_files": modality_files,
        "multiclass_mask_file": (
            multiclass_mask_path.relative_to(PROJECT_ROOT).as_posix()
        ),
        "whole_tumor_mask_file": (
            whole_tumor_mask_path.relative_to(PROJECT_ROOT).as_posix()
        ),
        "clinical_interpretation": (
            "Public de-identified research image linked to synthetic FHIR "
            "demo context. This is not a real longitudinal patient record."
        ),
    }
    metadata_path = case_image_root / "metadata.json"
    write_json(metadata_path, case_metadata)

    prepared_rows.append(
        {
            **case_metadata,
            "metadata_file": metadata_path.relative_to(PROJECT_ROOT).as_posix(),
            "image_mask_shape_match": True,
            "image_mask_affine_match": True,
            "mask_nonempty": True,
            "label_values_valid": True,
        }
    )

print("=" * 92)
print("✅ MRI modalities and reference masks standardized")
print(f"🧠 Prepared cases: {len(prepared_rows)}")
print(f"🧲 Modality files: {sum(len(row['modality_files']) for row in prepared_rows)}")
print(f"🎯 Whole-tumor masks: {len(prepared_rows)}")
print("=" * 92)

✅ MRI modalities and reference masks standardized
🧠 Prepared cases: 3
🧲 Modality files: 12
🎯 Whole-tumor masks: 3


In [8]:
# Cell 6 — Create previews and link imaging inputs to synthetic FHIR context

def robust_rescale(slice_data: np.ndarray) -> np.ndarray:
    finite = slice_data[np.isfinite(slice_data)]
    nonzero = finite[finite != 0]
    values = nonzero if nonzero.size else finite
    if values.size == 0:
        return np.zeros_like(slice_data, dtype=np.float32)
    lower, upper = np.percentile(values, [1.0, 99.0])
    if upper <= lower:
        upper = lower + 1.0
    scaled = np.clip((slice_data - lower) / (upper - lower), 0.0, 1.0)
    return scaled.astype(np.float32)

prepared_by_case = {row["case_id"]: row for row in prepared_rows}
manifest_cases: list[dict[str, Any]] = []
integrity_rows: list[dict[str, Any]] = []

for case_id in case_order:
    prepared = prepared_by_case[case_id]
    fhir_case = demo_by_id[case_id]

    flair_name = next(
        (
            name
            for name in prepared["modality_files"]
            if "flair" in name.lower()
        ),
        next(iter(prepared["modality_files"])),
    )
    flair_path = PROJECT_ROOT / prepared["modality_files"][flair_name]
    mask_path = PROJECT_ROOT / prepared["whole_tumor_mask_file"]

    flair = np.asanyarray(nib.load(str(flair_path)).dataobj)
    mask = np.asanyarray(nib.load(str(mask_path)).dataobj) > 0
    tumor_by_slice = mask.sum(axis=(0, 1))
    max_slice_index = int(np.argmax(tumor_by_slice))

    image_slice = robust_rescale(flair[:, :, max_slice_index])
    mask_slice = mask[:, :, max_slice_index]

    preview_path = PREVIEW_ROOT / f"{case_id}_mri_mask_preview.png"
    figure, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(np.rot90(image_slice), cmap="gray")
    axes[0].set_title(f"{case_id}: {flair_name}")
    axes[0].axis("off")

    axes[1].imshow(np.rot90(image_slice), cmap="gray")
    axes[1].imshow(
        np.rot90(mask_slice.astype(np.float32)),
        alpha=np.rot90(mask_slice.astype(np.float32)) * 0.45,
        cmap="autumn",
        vmin=0,
        vmax=1,
    )
    axes[1].set_title(
        f"Reference whole-tumor overlay\nslice {max_slice_index}"
    )
    axes[1].axis("off")
    figure.tight_layout()
    figure.savefig(preview_path, dpi=160, bbox_inches="tight")
    plt.close(figure)

    if not preview_path.exists() or preview_path.stat().st_size == 0:
        raise IOError(f"Preview was not created: {preview_path}")

    case_manifest_entry = {
        "case_id": case_id,
        "source_case_id": prepared["source_case_id"],
        "patient_reference": fhir_case["patient_reference"],
        "condition_reference": fhir_case["condition_reference"],
        "baseline_imaging_reference": (
            fhir_case["baseline_imaging_reference"]
        ),
        "followup_imaging_reference": (
            fhir_case["followup_imaging_reference"]
        ),
        "prior_observation_reference": (
            fhir_case["prior_observation_reference"]
        ),
        "baseline_volume_ml": fhir_case["baseline_volume_ml"],
        "planned_followup_reference_volume_ml": (
            fhir_case["planned_followup_reference_volume_ml"]
        ),
        "planned_percent_change": fhir_case["planned_percent_change"],
        "future_qc_expectation": fhir_case["future_qc_expectation"],
        "future_review_expectation": fhir_case["future_review_expectation"],
        "public_reference_volume_ml": (
            prepared["whole_tumor_reference_volume_ml"]
        ),
        "imaging_role": "followup-model-input",
        "baseline_context_source": "synthetic-prior-fhir-observation",
        "true_longitudinal_public_pair": False,
        "not_real_patient_linkage": True,
        "planned_future_perturbation": case_id == "low-confidence",
        "modality_files": prepared["modality_files"],
        "reference_multiclass_mask_file": (
            prepared["multiclass_mask_file"]
        ),
        "reference_whole_tumor_mask_file": (
            prepared["whole_tumor_mask_file"]
        ),
        "metadata_file": prepared["metadata_file"],
        "preview_file": preview_path.relative_to(PROJECT_ROOT).as_posix(),
        "source_dataset": DATASET_NAME,
        "source_license": DATASET_LICENSE,
        "source_citation_doi": DATASET_DOI,
    }
    manifest_cases.append(case_manifest_entry)

    integrity_rows.append(
        {
            "case_id": case_id,
            "source_case_id": prepared["source_case_id"],
            "spatial_shape": "x".join(
                str(value) for value in prepared["spatial_shape"]
            ),
            "modality_count": len(prepared["modality_files"]),
            "unique_labels": ",".join(
                str(value)
                for value in prepared["unique_reference_labels"]
            ),
            "whole_tumor_voxels": prepared["whole_tumor_voxels"],
            "whole_tumor_reference_volume_ml": (
                prepared["whole_tumor_reference_volume_ml"]
            ),
            "shape_match": prepared["image_mask_shape_match"],
            "affine_match": prepared["image_mask_affine_match"],
            "mask_nonempty": prepared["mask_nonempty"],
            "label_values_valid": prepared["label_values_valid"],
            "preview_created": True,
        }
    )

imaging_manifest = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "generated_utc": utc_now(),
    "notebook_number": "03",
    "dataset": {
        "name": DATASET_NAME,
        "bucket": AWS_BUCKET,
        "prefix": DATASET_PREFIX,
        "license": DATASET_LICENSE,
        "citation_doi": DATASET_DOI,
        "aws_registry_url": AWS_REGISTRY_URL,
        "medical_decathlon_url": MSD_URL,
    },
    "data_governance": {
        "public_deidentified_imaging_only": True,
        "synthetic_fhir_context_only": True,
        "real_patient_linkage_claimed": False,
        "true_longitudinal_public_pair_claimed": False,
        "clinical_use": False,
    },
    "interpretation": (
        "Each public image/mask pair is mapped to a synthetic FHIR demo case "
        "as a follow-up model input. The mapping is a research demonstration "
        "adapter, not identity linkage and not a claim of true longitudinal "
        "public-patient data."
    ),
    "cases": manifest_cases,
}
write_json(IMAGING_MANIFEST_PATH, imaging_manifest)

with IMAGING_INDEX_CSV_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    fieldnames = [
        "case_id",
        "source_case_id",
        "patient_reference",
        "followup_imaging_reference",
        "public_reference_volume_ml",
        "planned_followup_reference_volume_ml",
        "planned_future_perturbation",
        "metadata_file",
        "preview_file",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(
        {
            key: case[key]
            for key in fieldnames
        }
        for case in manifest_cases
    )

write_json(
    INTEGRITY_REPORT_PATH,
    {
        "generated_utc": utc_now(),
        "case_count": len(integrity_rows),
        "all_cases_passed": all(
            row["shape_match"]
            and row["affine_match"]
            and row["mask_nonempty"]
            and row["label_values_valid"]
            and row["preview_created"]
            and row["modality_count"] == 4
            for row in integrity_rows
        ),
        "cases": integrity_rows,
    },
)

with INTEGRITY_REPORT_CSV_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(integrity_rows[0]),
    )
    writer.writeheader()
    writer.writerows(integrity_rows)

print("=" * 92)
print("✅ Imaging-to-FHIR demonstration mapping created")
print(f"📋 Imaging manifest: {IMAGING_MANIFEST_PATH}")
print(f"🖼️ Previews created: {len(list(PREVIEW_ROOT.glob('*.png')))}")
print("⚠️ Mapping is synthetic context linkage, not real patient identity linkage")
print("=" * 92)

✅ Imaging-to-FHIR demonstration mapping created
📋 Imaging manifest: /content/drive/MyDrive/neurofhir-qc/data/sample_images/notebook_03/imaging_case_manifest.json
🖼️ Previews created: 3
⚠️ Mapping is synthetic context linkage, not real patient identity linkage


In [9]:
# Cell 7 — Create reusable imaging adapter, verification script, and documentation

IMAGING_ADAPTER_PATH = (
    PROJECT_ROOT / "backend/app/services/imaging_adapter.py"
)
VERIFY_SCRIPT_PATH = PROJECT_ROOT / "scripts/verify_imaging_data.py"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements/imaging.txt"
DOCUMENTATION_PATH = PROJECT_ROOT / "docs/IMAGING_DATA_PREPARATION.md"
OUTPUT_README_PATH = IMAGE_ROOT / "README.md"

imaging_adapter_source = r"""
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import nibabel as nib
import numpy as np


@dataclass(frozen=True)
class PreparedImagingCase:
    case_id: str
    source_case_id: str
    patient_reference: str
    followup_imaging_reference: str
    modality_files: dict[str, str]
    reference_whole_tumor_mask_file: str
    metadata_file: str
    public_reference_volume_ml: float
    planned_future_perturbation: bool


class ImagingAdapter:
    "Read the prepared public-imaging / synthetic-FHIR demo mapping."

    def __init__(self, project_root: Path, manifest_path: Path | None = None):
        self.project_root = Path(project_root)
        self.manifest_path = (
            Path(manifest_path)
            if manifest_path is not None
            else self.project_root
            / "data/sample_images/notebook_03/imaging_case_manifest.json"
        )
        with self.manifest_path.open("r", encoding="utf-8") as handle:
            self.manifest: dict[str, Any] = json.load(handle)
        self._cases = {
            case["case_id"]: case
            for case in self.manifest.get("cases", [])
        }

    def case_ids(self) -> list[str]:
        return sorted(self._cases)

    def get_case(self, case_id: str) -> PreparedImagingCase:
        try:
            case = self._cases[case_id]
        except KeyError as exc:
            raise KeyError(
                f"Unknown imaging case {case_id!r}; "
                f"available={self.case_ids()}"
            ) from exc
        return PreparedImagingCase(
            case_id=case["case_id"],
            source_case_id=case["source_case_id"],
            patient_reference=case["patient_reference"],
            followup_imaging_reference=case["followup_imaging_reference"],
            modality_files=dict(case["modality_files"]),
            reference_whole_tumor_mask_file=(
                case["reference_whole_tumor_mask_file"]
            ),
            metadata_file=case["metadata_file"],
            public_reference_volume_ml=float(
                case["public_reference_volume_ml"]
            ),
            planned_future_perturbation=bool(
                case["planned_future_perturbation"]
            ),
        )

    def load_modality(
        self,
        case_id: str,
        modality_name: str,
    ) -> tuple[np.ndarray, nib.Nifti1Image]:
        case = self.get_case(case_id)
        if modality_name not in case.modality_files:
            raise KeyError(
                f"Unknown modality {modality_name!r}; "
                f"available={sorted(case.modality_files)}"
            )
        path = self.project_root / case.modality_files[modality_name]
        image = nib.load(str(path))
        return np.asanyarray(image.dataobj), image

    def load_reference_mask(
        self,
        case_id: str,
    ) -> tuple[np.ndarray, nib.Nifti1Image]:
        case = self.get_case(case_id)
        path = self.project_root / case.reference_whole_tumor_mask_file
        image = nib.load(str(path))
        return np.asanyarray(image.dataobj), image

    def calculate_binary_mask_volume_ml(self, case_id: str) -> float:
        mask, image = self.load_reference_mask(case_id)
        spacing = image.header.get_zooms()[:3]
        voxel_volume_mm3 = float(np.prod(spacing))
        return float(np.count_nonzero(mask > 0) * voxel_volume_mm3 / 1000.0)
"""

verify_script_source = r"""
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def main() -> None:
    project_root = Path(
        os.getenv(
            "NEUROFHIR_QC_PROJECT_ROOT",
            "/content/drive/MyDrive/neurofhir-qc",
        )
    )
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    from backend.app.services.imaging_adapter import ImagingAdapter

    adapter = ImagingAdapter(project_root)
    results = []
    for case_id in adapter.case_ids():
        case = adapter.get_case(case_id)
        measured_volume = adapter.calculate_binary_mask_volume_ml(case_id)
        results.append(
            {
                "case_id": case_id,
                "source_case_id": case.source_case_id,
                "modality_count": len(case.modality_files),
                "reference_volume_ml_manifest": (
                    case.public_reference_volume_ml
                ),
                "reference_volume_ml_recomputed": measured_volume,
                "absolute_difference_ml": abs(
                    measured_volume - case.public_reference_volume_ml
                ),
            }
        )
    print(json.dumps(results, indent=2))


if __name__ == "__main__":
    main()
"""

documentation = f"""
# Imaging Data Preparation

Generated by `{NOTEBOOK_FILENAME}`.

## Public source

- Dataset: {DATASET_NAME}
- AWS bucket: `s3://{AWS_BUCKET}/{DATASET_PREFIX}`
- License: {DATASET_LICENSE}
- Citation DOI: `{DATASET_DOI}`

The AWS Open Data mirror exposes the task in uncompressed form. Notebook 03
downloads only a deterministic screening subset of labels and the three selected
four-modal MRI volumes, avoiding the full multi-gigabyte task archive.

## Prepared cases

The `stable`, `progression`, and `low-confidence` demonstration cases each
contain four 3D MRI modalities, a multiclass expert reference mask, a binary
whole-tumor reference mask, metadata, and a preview.

## Safety and interpretation

- The imaging data are public de-identified research data.
- FHIR patient context is synthetic.
- No real patient identity linkage is made.
- The public cases are not claimed to be true longitudinal image pairs.
- Reference-mask volume is a data-integrity/evaluation measurement, not a
  current AI result.
- No segmentation inference, QC classification, human review, or FHIR
  AI-result write-back occurs in Notebook 03.

## Main artifacts

- `data/sample_images/notebook_03/imaging_case_manifest.json`
- `data/sample_images/notebook_03/imaging_case_index.csv`
- `data/sample_masks/notebook_03/<case>/`
- `evaluation/results/notebook_03_imaging_preparation/`
- `backend/app/services/imaging_adapter.py`
- `scripts/verify_imaging_data.py`
"""

output_readme = f"""
# Notebook 03 Prepared Imaging Data

This directory contains the prepared public de-identified MRI inputs linked to
the three synthetic NeuroFHIR-QC demonstration contexts.

## Source

- {DATASET_NAME}
- License: {DATASET_LICENSE}
- DOI: {DATASET_DOI}

## Interpretation

The mapping is a research-demo adapter. It does not assert that a public-image
donor is the synthetic FHIR patient, and it does not assert that the public
images are genuine longitudinal scans of one person.

Use `imaging_case_manifest.json` as the canonical input index for later
segmentation and QC notebooks.
"""

IMAGING_ADAPTER_PATH.parent.mkdir(parents=True, exist_ok=True)
VERIFY_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
REQUIREMENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
DOCUMENTATION_PATH.parent.mkdir(parents=True, exist_ok=True)

IMAGING_ADAPTER_PATH.write_text(
    textwrap.dedent(imaging_adapter_source).strip() + "\n",
    encoding="utf-8",
)
VERIFY_SCRIPT_PATH.write_text(
    textwrap.dedent(verify_script_source).strip() + "\n",
    encoding="utf-8",
)
REQUIREMENTS_PATH.write_text(
    "\n".join(
        [
            f"boto3=={dependency_versions['boto3']}",
            f"botocore=={dependency_versions['botocore']}",
            f"nibabel=={dependency_versions['nibabel']}",
            f"numpy=={dependency_versions['numpy']}",
            f"pandas=={dependency_versions['pandas']}",
            f"matplotlib=={dependency_versions['matplotlib']}",
        ]
    )
    + "\n",
    encoding="utf-8",
)
DOCUMENTATION_PATH.write_text(
    textwrap.dedent(documentation).strip() + "\n",
    encoding="utf-8",
)
OUTPUT_README_PATH.write_text(
    textwrap.dedent(output_readme).strip() + "\n",
    encoding="utf-8",
)

compile(
    IMAGING_ADAPTER_PATH.read_text(encoding="utf-8"),
    str(IMAGING_ADAPTER_PATH),
    "exec",
)
compile(
    VERIFY_SCRIPT_PATH.read_text(encoding="utf-8"),
    str(VERIFY_SCRIPT_PATH),
    "exec",
)

reusable_files = [
    IMAGING_ADAPTER_PATH,
    VERIFY_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    DOCUMENTATION_PATH,
    OUTPUT_README_PATH,
]
missing_reusable = [
    str(path)
    for path in reusable_files
    if not path.exists() or path.stat().st_size == 0
]
if missing_reusable:
    raise FileNotFoundError(
        "Reusable imaging files are missing:\n"
        + "\n".join(f" - {path}" for path in missing_reusable)
    )

print("=" * 92)
print("✅ Reusable imaging adapter and verification script created")
print("✅ Generated Python files passed syntax compilation")
print(f"📦 Imaging dependencies recorded: {REQUIREMENTS_PATH}")
print("=" * 92)

✅ Reusable imaging adapter and verification script created
✅ Generated Python files passed syntax compilation
📦 Imaging dependencies recorded: /content/drive/MyDrive/neurofhir-qc/requirements/imaging.txt


In [10]:
# Cell 8 — Re-load all prepared artifacts and measure integrity

# Import the generated adapter without requiring a package installation.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.services.imaging_adapter import ImagingAdapter

adapter = ImagingAdapter(PROJECT_ROOT)
if adapter.case_ids() != sorted(case_order):
    raise AssertionError(
        f"Unexpected adapter cases: {adapter.case_ids()}"
    )

adapter_validation_rows: list[dict[str, Any]] = []
for case_id in case_order:
    prepared_case = adapter.get_case(case_id)
    if len(prepared_case.modality_files) != 4:
        raise AssertionError(f"{case_id} does not have four modalities.")

    modality_shapes = {}
    modality_affines = {}
    for modality_name in sorted(prepared_case.modality_files):
        array, image = adapter.load_modality(case_id, modality_name)
        modality_shapes[modality_name] = tuple(int(v) for v in array.shape)
        modality_affines[modality_name] = image.affine

    if len(set(modality_shapes.values())) != 1:
        raise AssertionError(
            f"Modality shape mismatch for {case_id}: {modality_shapes}"
        )
    first_affine = next(iter(modality_affines.values()))
    if not all(
        np.allclose(first_affine, affine, atol=1e-4, rtol=0.0)
        for affine in modality_affines.values()
    ):
        raise AssertionError(f"Modality affine mismatch for {case_id}.")

    mask, mask_image = adapter.load_reference_mask(case_id)
    if tuple(mask.shape) != next(iter(modality_shapes.values())):
        raise AssertionError(f"Mask shape mismatch for {case_id}.")
    if not np.allclose(
        first_affine,
        mask_image.affine,
        atol=1e-4,
        rtol=0.0,
    ):
        raise AssertionError(f"Mask affine mismatch for {case_id}.")
    if int(np.count_nonzero(mask > 0)) <= 0:
        raise AssertionError(f"Mask is empty for {case_id}.")

    recomputed_volume = adapter.calculate_binary_mask_volume_ml(case_id)
    volume_difference = abs(
        recomputed_volume - prepared_case.public_reference_volume_ml
    )
    if volume_difference > 1e-5:
        raise AssertionError(
            f"Reference-volume mismatch for {case_id}: {volume_difference}"
        )

    adapter_validation_rows.append(
        {
            "case_id": case_id,
            "source_case_id": prepared_case.source_case_id,
            "modality_count": len(prepared_case.modality_files),
            "spatial_shape": list(next(iter(modality_shapes.values()))),
            "mask_voxel_count": int(np.count_nonzero(mask > 0)),
            "reference_volume_ml": round(recomputed_volume, 6),
            "manifest_volume_difference_ml": round(volume_difference, 9),
            "shape_integrity": True,
            "affine_integrity": True,
            "volume_integrity": True,
        }
    )

write_json(
    EVALUATION_ROOT / "adapter_validation_report.json",
    {
        "generated_utc": utc_now(),
        "all_cases_passed": True,
        "cases": adapter_validation_rows,
    },
)

expected_counts = {
    "prepared_cases": 3,
    "modality_files": 12,
    "multiclass_masks": 3,
    "whole_tumor_masks": 3,
    "previews": 3,
}
observed_counts = {
    "prepared_cases": len(adapter_validation_rows),
    "modality_files": sum(
        len(adapter.get_case(case_id).modality_files)
        for case_id in adapter.case_ids()
    ),
    "multiclass_masks": len(
        list(MASK_ROOT.glob("*/reference_multiclass.nii.gz"))
    ),
    "whole_tumor_masks": len(
        list(MASK_ROOT.glob("*/whole_tumor_binary.nii.gz"))
    ),
    "previews": len(list(PREVIEW_ROOT.glob("*.png"))),
}
if observed_counts != expected_counts:
    raise AssertionError(
        f"Prepared-artifact counts failed: {observed_counts}"
    )

print("=" * 92)
print("✅ Prepared imaging artifacts reloaded successfully")
print("✅ 3/3 case mappings passed shape, affine, mask, and volume checks")
print("✅ 12/12 modality files are available")
print(f"📊 Observed counts: {observed_counts}")
print("=" * 92)

✅ Prepared imaging artifacts reloaded successfully
✅ 3/3 case mappings passed shape, affine, mask, and volume checks
✅ 12/12 modality files are available
📊 Observed counts: {'prepared_cases': 3, 'modality_files': 12, 'multiclass_masks': 3, 'whole_tumor_masks': 3, 'previews': 3}


In [11]:
# Cell 9 — Create the final audit and update the notebook manifest

core_files = [
    DATASET_METADATA_PATH,
    SOURCE_INVENTORY_PATH,
    SELECTION_REPORT_PATH,
    SELECTION_REPORT_CSV_PATH,
    IMAGING_MANIFEST_PATH,
    IMAGING_INDEX_CSV_PATH,
    INTEGRITY_REPORT_PATH,
    INTEGRITY_REPORT_CSV_PATH,
    EVALUATION_ROOT / "adapter_validation_report.json",
    IMAGING_ADAPTER_PATH,
    VERIFY_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    DOCUMENTATION_PATH,
    OUTPUT_README_PATH,
]
core_files.extend(sorted(IMAGE_ROOT.glob("*/metadata.json")))
core_files.extend(sorted(IMAGE_ROOT.glob("*/modalities/*.nii.gz")))
core_files.extend(sorted(MASK_ROOT.glob("*/*.nii.gz")))
core_files.extend(sorted(PREVIEW_ROOT.glob("*.png")))

missing_core = [
    str(path)
    for path in core_files
    if not path.exists() or path.stat().st_size == 0
]
if missing_core:
    raise FileNotFoundError(
        "Notebook 03 evidence is missing:\n"
        + "\n".join(f" - {path}" for path in missing_core)
    )

integrity_report = load_json(INTEGRITY_REPORT_PATH)
if not integrity_report.get("all_cases_passed", False):
    raise AssertionError("Notebook 03 imaging integrity did not pass.")

NOTEBOOK_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
candidates = [
    NOTEBOOK_SAVE_PATH,
    PROJECT_ROOT / NOTEBOOK_FILENAME,
    PROJECT_ROOT / "03_NeuroFHIR_QC_Imaging_Data_Preparation.ipynb",
]
candidates.extend(sorted(PROJECT_ROOT.glob("*03*.ipynb")))
candidates.extend(
    sorted((PROJECT_ROOT / "notebooks").glob("*03*.ipynb"))
)
found = next(
    (
        path
        for path in candidates
        if path.exists() and path.is_file() and path.stat().st_size > 0
    ),
    None,
)
if found and found.resolve() != NOTEBOOK_SAVE_PATH.resolve():
    shutil.copy2(found, NOTEBOOK_SAVE_PATH)

notebook_saved = (
    NOTEBOOK_SAVE_PATH.exists()
    and NOTEBOOK_SAVE_PATH.stat().st_size > 0
)
manifest_status = (
    "completed"
    if notebook_saved
    else "executed_pending_notebook_save"
)

checksum_paths = sorted(
    {
        path.resolve()
        for path in core_files
        if path.exists() and path.is_file()
    },
    key=lambda path: str(path),
)
checksums = [
    {
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in checksum_paths
]

metrics = {
    "public_source_pairs_discovered": source_inventory["paired_case_count"],
    "candidate_reference_masks_screened": len(candidate_summaries),
    "prepared_case_count": observed_counts["prepared_cases"],
    "prepared_case_success_rate": 1.0,
    "modality_file_count": observed_counts["modality_files"],
    "expected_modalities_per_case": 4,
    "multiclass_reference_mask_count": observed_counts["multiclass_masks"],
    "whole_tumor_reference_mask_count": observed_counts["whole_tumor_masks"],
    "preview_count": observed_counts["previews"],
    "shape_integrity_rate": 1.0,
    "affine_integrity_rate": 1.0,
    "nonempty_reference_mask_rate": 1.0,
    "label_value_validation_rate": 1.0,
    "adapter_reload_success_rate": 1.0,
}

audit = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "notebook_number": "03",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": manifest_status,
    "audited_utc": utc_now(),
    "notebook_saved_to_project": notebook_saved,
    "required_notebook_save_path": str(NOTEBOOK_SAVE_PATH),
    "dataset": {
        "name": DATASET_NAME,
        "bucket": AWS_BUCKET,
        "prefix": DATASET_PREFIX,
        "license": DATASET_LICENSE,
        "citation_doi": DATASET_DOI,
        "public_deidentified_research_data": True,
    },
    "data_governance": {
        "synthetic_fhir_context_only": True,
        "real_patient_linkage_claimed": False,
        "true_longitudinal_public_pair_claimed": False,
        "phi_allowed": False,
        "clinical_use": False,
    },
    "scope": {
        "public_mri_prepared": True,
        "reference_masks_prepared": True,
        "imaging_fhir_demo_mapping_created": True,
        "segmentation_inference_run": False,
        "current_ai_observation_created": False,
        "qc_classification_calculated": False,
        "human_review_transition_executed": False,
        "ai_result_writeback_performed": False,
    },
    "metrics": metrics,
    "checksum_inventory": checksums,
    "next_notebook": (
        "04 — Segmentation and Volumetry, only after Notebook 03 "
        "is marked completed."
    ),
}
write_json(AUDIT_JSON_PATH, audit)

audit_markdown = f"""
# Notebook 03 — Imaging Data Preparation

**Notebook:** `{NOTEBOOK_FILENAME}`
**Status:** `{manifest_status}`
**Dataset:** {DATASET_NAME}
**License:** {DATASET_LICENSE}
**Audited UTC:** {audit['audited_utc']}

## Evidence

| Metric | Result |
|---|---:|
| Public paired cases discovered | {metrics['public_source_pairs_discovered']} |
| Candidate masks screened | {metrics['candidate_reference_masks_screened']} |
| Demo cases prepared | {metrics['prepared_case_count']}/3 |
| MRI modality files | {metrics['modality_file_count']}/12 |
| Multiclass reference masks | {metrics['multiclass_reference_mask_count']}/3 |
| Whole-tumor binary masks | {metrics['whole_tumor_reference_mask_count']}/3 |
| Visual previews | {metrics['preview_count']}/3 |
| Shape integrity | {metrics['shape_integrity_rate']:.1%} |
| Affine integrity | {metrics['affine_integrity_rate']:.1%} |
| Non-empty mask rate | {metrics['nonempty_reference_mask_rate']:.1%} |
| Label-value validation | {metrics['label_value_validation_rate']:.1%} |
| Adapter reload success | {metrics['adapter_reload_success_rate']:.1%} |

## Interpretation boundary

The MRI data are public de-identified research images. The FHIR patient
contexts are synthetic. The mapping does not identify a public image donor as a
synthetic patient and does not claim a true longitudinal public-image pair.

Notebook 03 measured reference-mask volume only for data integrity and later
segmentation evaluation. It did not run a segmentation model, create a current
AI Observation, calculate the final QC class, perform human review, or write an
AI result to FHIR.

## Completion gate

Required notebook path:

`{NOTEBOOK_SAVE_PATH}`

Notebook detected there: **{'yes' if notebook_saved else 'no'}**

If no, save this notebook there and rerun Cell 9.
"""

AUDIT_MD_PATH.parent.mkdir(parents=True, exist_ok=True)
AUDIT_MD_PATH.write_text(
    textwrap.dedent(audit_markdown).strip() + "\n",
    encoding="utf-8",
)

notebook_03_entry["status"] = manifest_status
notebook_03_entry["last_executed_utc"] = utc_now()
notebook_03_entry["dataset_name"] = DATASET_NAME
notebook_03_entry["prepared_case_count"] = metrics["prepared_case_count"]
notebook_03_entry["modality_file_count"] = metrics["modality_file_count"]
notebook_03_entry["imaging_integrity_rate"] = 1.0
notebook_03_entry["audit_path"] = (
    AUDIT_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()
)
write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

print("=" * 92)
print("✅ Notebook 03 imaging preparation evidence passed")
print("✅ 3/3 public MRI/reference-mask case packages prepared")
print("✅ 12/12 modality files validated")
print("✅ 3/3 whole-tumor masks and previews validated")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print(f"✅ Audit Markdown: {AUDIT_MD_PATH}")
print(f"📓 Manifest status: {manifest_status}")
if notebook_saved:
    print("🎯 Notebook 03 is complete; Notebook 04 may begin")
else:
    print("⚠️ Save this notebook into the required Drive path")
    print(f"   {NOTEBOOK_SAVE_PATH}")
    print("Then rerun Cell 9 before beginning Notebook 04.")
print("=" * 92)

✅ Notebook 03 imaging preparation evidence passed
✅ 3/3 public MRI/reference-mask case packages prepared
✅ 12/12 modality files validated
✅ 3/3 whole-tumor masks and previews validated
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_03_imaging_preparation_audit.json
✅ Audit Markdown: /content/drive/MyDrive/neurofhir-qc/docs/NOTEBOOK_03_IMAGING_DATA_PREPARATION.md
📓 Manifest status: executed_pending_notebook_save
⚠️ Save this notebook into the required Drive path
   /content/drive/MyDrive/neurofhir-qc/notebooks/03_NeuroFHIR_QC_Imaging_Data_Preparation.ipynb
Then rerun Cell 9 before beginning Notebook 04.
